In [ ]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import pandas as pd
import polars as pl
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from sklearn.cluster import DBSCAN, HDBSCAN, OPTICS
from sklearn.neighbors import KDTree
from sklearn.neighbors import NearestNeighbors

import sys


from pyS3M import IOFunctions

IO = IOFunctions.IO_Functions()

from pyS3M import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from pyS3M import PlottingBase

plotter = PlottingBase.PublicationPlotter(dark_background=False)

from pyS3M import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from pyS3M import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from pyS3M import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from pyS3M import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs(camera="zwo")

from pyS3M import MaskFunctions

M_F = MaskFunctions.Mask_Functions(camera="zwo")

from pyS3M import SpotDetectionFunctions

SD_F = SpotDetectionFunctions.SpotDetection_Functions(camera="zwo")

from pyS3M import SR_Functions

SupRes_F = SR_Functions.SuperRes_Functions(camera="zwo")

from pyS3M import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

from pyS3M import render
from pyS3M import FiducialDetection

FD = FiducialDetection.FiducialDetector()


import types

import pyS3M.DriftCorrectionFunctions as DCF
from pyS3M.LinkingFunctions import link_localisations


smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [ ]:
data_folder = "../../Camera_Calibrations/ZWO_Camera"
gain_map = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset_map = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
read_noise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

pixel_QYs = np.vstack([B, G, R])
camera_parameters = {}
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ["B", "G", "R"]
camera_parameters["pixel_order_indices"] = [0, 1, 2]

In [ ]:
image_folder = "/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/Brendan/20260623_MASSIVECELLS/ATTO655_CF640R/250pM_F1_CF640R_750pM_F2_ATTO655_30mW637_1"

In [ ]:
localisation_files = H_F.file_search(image_folder, ".h5", "")

In [ ]:
localisation_files

In [ ]:
tif_files = H_F.file_search(image_folder, ".tif", "")

In [ ]:
metadata_files = H_F.file_search(image_folder, "metadata", "")

In [ ]:
x_coord, y_coord, width, height = IO.metadata_reader_imageJ(metadata_files[0])
n_frames = IO.metadata_nframes_reader_imageJ(metadata_files[0])

In [ ]:
loc_data = pd.read_hdf(localisation_files[0])

In [ ]:
plt.hist(loc_data['A_R'], 100);
plt.xlim([0, 1])
plt.show()

In [ ]:
loc_data = loc_data[loc_data["xc_err"] < 60/72]
loc_data = loc_data[loc_data["yc_err"] < 60/72]

loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 60/72]
loc_data = loc_data[loc_data["s_y_err"] < 60/72]

loc_data = loc_data[(loc_data["s_x"] > 100/72) & (loc_data["s_x"] < 200/72)]
loc_data = loc_data[(loc_data["s_y"] > 100/72) & (loc_data["s_y"] < 200/72)]

loc_data = loc_data[loc_data["A_B_err"] < 0.4]
loc_data = loc_data[loc_data["A_G_err"] < 0.4]
loc_data = loc_data[loc_data["A_R_err"] < 0.4]

loc_data = loc_data[loc_data["A_B"] > 0.005]
loc_data = loc_data[loc_data["A_G"] > 0.05]
loc_data = loc_data[loc_data["A_R"] > 0.5]

loc_data = loc_data[loc_data["bg_B_err"] < 0.15]
loc_data = loc_data[loc_data["bg_G_err"] < 0.15]
loc_data = loc_data[loc_data["bg_R_err"] < 0.15]

loc_data = loc_data[~((loc_data['photons'] < 500) & (loc_data['spot_matched_filter_response'] < 25))]

loc_data = loc_data[loc_data["spot_background_std"] > 20]

#loc_data = loc_data[loc_data["photons"] < 100000]
#loc_data = loc_data[loc_data["photons"] > 100]


loc_data = loc_data[loc_data["chi_sqr"] < 2.5]

In [ ]:
plt.hist(loc_data['A_R'], 1000, alpha=0.5, color='darkred');
plt.hist(loc_data['A_G'], 1000, alpha=0.5, color='darkgreen');
#plt.hist(loc_data['A_B'], 1000, alpha=0.5, color='lightblue');
plt.xlim([0, 1])
plt.show()

In [ ]:
image = render.render(
    locs=loc_data.to_records(index=False),
    oversampling=1,
    viewport=((0, 0), (height, width)),
    blur_method="smooth",
)[1]

In [ ]:
fig, axs = plotter.two_column_plot()

axs = plotter.image_plot(axs, image, colorbar=False)
plt.show()

In [ ]:
drift_corrector = DCF.Drift_Correction_Functions()

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(loc_data['frame']),      # Total number of frames in the movie
    "Pixelsize": 72,        # pixel size always in nm
}]

corrected_locs, drift_result = drift_corrector.undrift(locs=loc_data.to_records(index=False),
                                          info=info,
                                          method="aim",
                                          segmentation=20,
                                            intersect_d=20/72,
                                          roi_r=60/72,
                                         )
corrected_locs = pd.DataFrame(corrected_locs)

In [ ]:
plt.plot(drift_result.drift_x*72)
plt.plot(drift_result.drift_y*72)

plt.show()

In [ ]:
IO._write_h5_database(df=corrected_locs, filepath=localisation_files[0].split('.h5')[0]+'_Undrifted.h5', normalise_photons=False)

In [ ]:
link_r = 0.5 * (np.median(corrected_locs['xc_err']) + np.median(corrected_locs['yc_err']))
linked = link_localisations(corrected_locs, n_frames=n_frames, r_max=link_r, max_dark_time=2)

In [ ]:
oversampling = 1

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(loc_data['frame']),      # Total number of frames in the movie
    "Pixelsize": 72,        # pixel size always in nm
}]

image = render.render(
            locs=linked.to_records(index=False),
            info=info,
            blur_method="smooth",
            oversampling=oversampling
        )[1]  # Take the rendered image

In [ ]:
plt.imshow(image, origin='lower', vmax=np.percentile(image, 99))

In [ ]:
oversampling = 1

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(loc_data['frame']),      # Total number of frames in the movie
    "Pixelsize": 72,        # pixel size always in nm
}]

image = render.render(
            locs=linked.to_records(index=False),
            info=info,
            blur_method="smooth",
            oversampling=oversampling
        )[1]  # Take the rendered image

region_centers, binary_mask, threshold, metadata = drift_corrector.detect_high_density_regions_from_image(
      smoothed_image=image,
      histogram_bins=1024,
      threshold_percentile=99.99,
      pixelsize=69.0,
      title="Fiducial Detection"
  )

In [ ]:
info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(linked['frame']),      # Total number of frames in the movie
    "Pixelsize": 72,        # pixel size always in nm
}]

selected_puncta, selection_metadata = drift_corrector.select_puncta_from_regions(
        locs=linked.to_records(index=False),                             # Your localization data
        region_centres=region_centers,         # Output from Step 1
        binary_mask=binary_mask,               # Output from Step 1
        pixelsize=info[0]['Pixelsize'],
        selection_box_size_nm=350.0,         # Box size around each region center (adjust as needed)
        min_localisations_per_region=int(0.075*info[0]['Frames']),      # Minimum locs for valid fiducial (adjust as needed)
        title="Step 2: Puncta Selection",
        create_plot=True
    )

In [ ]:
linked_fremoved = FD.remove_puncta_locs(linked.to_records(index=False), selected_puncta)
linked_fremoved = pd.DataFrame(linked_fremoved)

In [ ]:
IO._write_h5_database(df=linked_fremoved, filepath=localisation_files[0].split('.h5')[0]+'_Undrifted_Linked_FiducialsRemoved.h5', normalise_photons=False)

In [ ]:
linked_fremoved = pd.read_hdf(localisation_files[1])

In [ ]:
oversampling = 8

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(linked_fremoved['frame']),      # Total number of frames in the movie
    "Pixelsize": 72,        # pixel size always in nm
}]

image = render.render(
            locs=linked_fremoved.to_records(index=False),
            info=info,
            blur_method="gaussian",
            oversampling=oversampling
        )[1]  # Take the rendered image

In [ ]:
fig, axs = plotter.two_column_plot(height=5, width=5)

axs = plotter.image_plot(axs, image, cmap='hot', colorbar='off')

plt.show()

## Site-pooled channel unmixing

Replaces the joint spatial-spectral clustering attempt below-turned-above
(`unmix_channels_joint_cluster`), which produced two clusters that didn't
correspond to real populations for this dye pair. Full diagnosis and
part-by-part validation against this dataset in `claude/Close_Red_Dyes.md`.

**Core idea**: ATTO 655 and CF640R are close enough (17 nm peak gap) that a
single blink's spectral ratio is dominated by shot noise — using it to gate
*clustering* (as the joint method does) corrupts cluster formation broadly.
Instead: cluster purely spatially first (whole movie, no spectral or
time-gap term — a genuine DNA-PAINT site rebinds many times over the whole
acquisition), pool each site's spectrum across all its repeat visits (shot
noise shrinks ~1/√N), then classify **sites**, never individual blinks, by
dye.

No single-dye control sample is available for this preparation, so the two
reference means are calibrated from a free 2-component fit on the pooled
data itself rather than from theory — using the theory-predicted means
directly as fixed anchors produced a badly degenerate covariance fit (see
`Close_Red_Dyes.md` §11); the empirically-fit means don't have this problem
and are what's used below.

**Revision 2 caveat (`Close_Red_Dyes.md` §14)**: running this against the
real data shows a bimodal split does develop, but the rendered result looks
*spatially segregated* into zones rather than interleaved — suspected
artefact of forcing one dye label per whole DBSCAN site regardless of how
generously `epsilon_multiplier` merges nearby structure. The eps-sweep
diagnostic a few cells down is the first, cheap test of that hypothesis
(§14 §5 item 1) before committing to a bigger kinetic-first redesign.

Requires `linked_fremoved` (temporally-linked, fiducial-removed) from the
cells above.

In [ ]:
# Site-pooled unmixing as a function of epsilon_multiplier AND clustering
# method, so the same pipeline can be run once for the default DBSCAN eps,
# swept over several eps values, and compared against HDBSCAN below
# (Close_Red_Dyes.md §14 §5's eps-sweep diagnostic, extended per the user's
# HDBSCAN suggestion after the eps sweep alone didn't fix the spatial
# segregation issue).
from pyS3M import SM_extractionfunctions
from pyS3M.clustering.hdbscan_clusterer import _get_hdbscan_cls
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from scipy.stats import multivariate_normal

SM_E = SM_extractionfunctions.extract_SMs(camera="zwo")


def run_site_pooled_unmixing(
    linked_fremoved, epsilon_multiplier=1.0, min_cluster_size=5, conf_threshold=0.90,
    clustering_method="dbscan", verbose=True,
):
    """Steps 2-6 of Close_Red_Dyes.md's site-pooled unmixing.

    clustering_method: 'dbscan' (epsilon_multiplier scales a single global
    radius) or 'hdbscan' (locally-adaptive density -- epsilon_multiplier is
    ignored; cluster_selection_epsilon is fixed at loc_precision, matching
    extract_single_molecules_HDBSCAN's own default).

    Returns (channel_0, channel_1, sites, diagnostics) where channel_0 =
    ATTO655-like (lower A_R), channel_1 = CF640R-like (higher A_R).
    """
    # Step 2: whole-movie spatial site clustering (xc, yc only).
    if clustering_method == "dbscan":
        sites, blinks = SM_E.extract_single_molecules_DBSCAN(
            linked_fremoved, min_cluster_size=min_cluster_size, epsilon_multiplier=epsilon_multiplier,
        )
    elif clustering_method == "hdbscan":
        sites, blinks = SM_E.extract_single_molecules_HDBSCAN(
            linked_fremoved, min_cluster_size=min_cluster_size,
        )
    else:
        raise ValueError(f"unknown clustering_method: {clustering_method!r}")

    # Step 3/4: classify sites against an empirically-calibrated fixed-mean model.
    X_sites = sites[['A_R', 'A_G']].to_numpy()
    gmm_free = GaussianMixture(
        n_components=2, covariance_type='full', random_state=42, n_init=10,
    ).fit(X_sites)
    order = np.argsort(gmm_free.means_[:, 0])
    fixed_means = gmm_free.means_[order]

    site_cov, site_weights, converged = SM_E.fit_covariances_fixed_means(
        X_sites, fixed_means, fit_type="EM", max_iter=200,
    )
    log_probs = np.zeros((len(X_sites), 2))
    for k in range(2):
        mvn = multivariate_normal(mean=fixed_means[k], cov=site_cov[k])
        log_probs[:, k] = mvn.logpdf(X_sites) + np.log(site_weights[k])
    log_probs -= log_probs.max(axis=1, keepdims=True)
    site_probs = np.exp(log_probs)
    site_probs /= site_probs.sum(axis=1, keepdims=True)
    sites = sites.copy()
    sites['dye_assignment'] = site_probs.argmax(axis=1)
    sites['dye_confidence'] = site_probs.max(axis=1)

    # Step 5: propagate site labels to every blink event.
    blinks_labelled = blinks.merge(
        sites[['molecular_index', 'dye_assignment', 'dye_confidence']],
        on='molecular_index', how='left',
    )

    # Step 6: isolated blinks -- same clustering method/params as Step 2, so
    # "isolated" consistently means "not in a site by this method". Classified
    # individually with covariance re-fit to the isolated (single-blink)
    # population -- NOT the pooled site covariance (Close_Red_Dyes.md §13's
    # lesson: reusing pooled covariance for single blinks gives spuriously
    # overconfident calls).
    quality = SM_E.filter_quality_localisations(loc_data=linked_fremoved)
    loc_precision = 0.5 * (quality['xc_err'].mean() + quality['yc_err'].mean())
    Xq = np.vstack([quality['xc'], quality['yc']]).T
    if clustering_method == "dbscan":
        labels_all = DBSCAN(
            min_samples=min_cluster_size, eps=loc_precision * epsilon_multiplier,
        ).fit(Xq).labels_
    else:
        HDBSCAN = _get_hdbscan_cls()
        labels_all = HDBSCAN(
            min_cluster_size=min_cluster_size, cluster_selection_epsilon=loc_precision,
        ).fit(Xq).labels_
    isolated = quality.loc[labels_all == -1].reset_index(drop=True)

    X_iso = isolated[['A_R', 'A_G']].to_numpy()
    iso_cov, iso_weights, _ = SM_E.fit_covariances_fixed_means(
        X_iso, fixed_means, fit_type="EM", max_iter=200,
    )
    log_probs = np.zeros((len(X_iso), 2))
    for k in range(2):
        mvn = multivariate_normal(mean=fixed_means[k], cov=iso_cov[k])
        log_probs[:, k] = mvn.logpdf(X_iso) + np.log(iso_weights[k])
    log_probs -= log_probs.max(axis=1, keepdims=True)
    iso_probs = np.exp(log_probs)
    iso_probs /= iso_probs.sum(axis=1, keepdims=True)
    isolated = isolated.copy()
    isolated['dye_assignment'] = iso_probs.argmax(axis=1)
    isolated['dye_confidence'] = iso_probs.max(axis=1)
    isolated_accepted = isolated[isolated['dye_confidence'] >= conf_threshold]

    # Combine into final per-dye tables.
    cols = ['xc', 'yc', 'xc_err', 'yc_err', 'dye_assignment', 'dye_confidence']
    combined = pd.concat([
        blinks_labelled[cols].dropna(subset=['dye_assignment']),
        isolated_accepted[cols],
    ], ignore_index=True)
    combined['dye_assignment'] = combined['dye_assignment'].astype(int)

    channel_0 = combined[combined['dye_assignment'] == 0].reset_index(drop=True)
    channel_1 = combined[combined['dye_assignment'] == 1].reset_index(drop=True)

    diagnostics = {
        'clustering_method': clustering_method,
        'epsilon_multiplier': epsilon_multiplier,
        'n_sites': len(sites),
        'fixed_means': fixed_means,
        'site_weights': site_weights,
        'n_channel_0': len(channel_0),
        'n_channel_1': len(channel_1),
    }
    if verbose:
        print(f"[{clustering_method}] eps_mult={epsilon_multiplier}: {len(sites):,} sites, "
              f"weights={site_weights}, channel_0={len(channel_0):,}, channel_1={len(channel_1):,}")

    return channel_0, channel_1, sites, diagnostics


In [ ]:
# Default run (epsilon_multiplier=1.0) -- kept as channel_0/channel_1 for
# compatibility with the render/plot/zoom-in cells below.
channel_0, channel_1, sites, diag = run_site_pooled_unmixing(linked_fremoved, epsilon_multiplier=0.5)


In [ ]:
oversampling = 8
pixel_size = 72

info = [{
    "Width": width,         # Image width in pixels
    "Height": height,        # Image height in pixels
    "Frames": np.max(linked_fremoved['frame']),      # Total number of frames in the movie
    "Pixelsize": pixel_size,        # pixel size always in nm
}]

min_pixel = 4
min_size = min_pixel*(pixel_size/1000)
# Render two channels
_, img1 = render.render(channel_0.to_records(index=False), info, oversampling=oversampling, blur_method='gaussian', min_blur_width=min_size)
_, img2 = render.render(channel_1.to_records(index=False), info, oversampling=oversampling, blur_method='gaussian', min_blur_width=min_size)


In [ ]:
fig, ax = plotter.one_column_plot(height=5, width=5)

ax = plotter.multichannel_overlay_plot(
  ax, [img2, img1],
  cmaps=['hot', 'cyan'],
  pixelsize=(pixel_size/oversampling),
  scalebarsize=10000,
  vmins=[np.percentile(img2, 1), np.percentile(img1, 1)],
  vmaxs=[np.percentile(img2, 99.9), np.percentile(img1, 99.9)],
  brightness_boost=[1, 1],
  scalebarlabel='10 μm'
)
import matplotlib.patches as patches

# xmin = 275
# ymin = 325
# rect = patches.Rectangle(
#     (xmin * oversampling, ymin * oversampling),
#     150 * oversampling,
#     150 * oversampling,
#     linewidth=0.5,
#     edgecolor="white",
#     facecolor="none",
# )

# # Add the patch to the Axes
# ax.add_patch(rect)

# ymin = 105
# xmin = 525
# width = 160
# height = int(width/2.12)
# rect = patches.Rectangle(
#     (ymin * oversampling, xmin * oversampling),
#     width * oversampling,
#     height * oversampling,
#     linewidth=0.5,
#     edgecolor="cyan",
#     facecolor="none",
# )
# ax.add_patch(rect)



folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, 'Massive_Cells_Render_Box_tworeds.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
# 2.12 ratio

# Create overlay
fig, ax = plotter.one_column_plot(height=2/2.12, width=2)

ymin = 105
xmin = 525
width = 160
height = int(width/2.12)
img1_subset = img1[int(xmin * oversampling):int((xmin+height) * oversampling), int(ymin * oversampling):int((ymin+width) * oversampling)]
img2_subset = img2[int(xmin * oversampling):int((xmin+height) * oversampling), int(ymin * oversampling):int((ymin+width) * oversampling)]


ax = plotter.multichannel_overlay_plot(
  ax, [img2_subset, img1_subset],
  cmaps=['hot', 'cyan'],
  pixelsize=(pixel_size/oversampling),
  scalebarsize=1000,
  vmins=[np.percentile(img2, 1), np.percentile(img1, 1)],
  vmaxs=[np.percentile(img2, 99.9), np.percentile(img1, 99.9)],
  brightness_boost=[1, 1],
  scalebarlabel='1 μm'
)


folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, 'Massive_Cells_Render_Zoomin_Mitochondria.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
# Create overlay
fig, ax = plotter.one_column_plot(height=2, width=2)

ymin = 275
xmin = 325
img1_subset = img1[int(xmin * oversampling):int((xmin+150) * oversampling), int(ymin * oversampling):int((ymin+150) * oversampling)]
img2_subset = img2[int(xmin * oversampling):int((xmin+150) * oversampling), int(ymin * oversampling):int((ymin+150) * oversampling)]


ax = plotter.multichannel_overlay_plot(
  ax, [img2_subset, img1_subset],
  cmaps=['hot', 'cyan'],
  pixelsize=(pixel_size/oversampling),
  scalebarsize=1000,
  vmins=[np.percentile(img2, 1), np.percentile(img1, 1)],
  vmaxs=[np.percentile(img2, 99.9), np.percentile(img1, 99.9)],
  brightness_boost=[1, 1],
  scalebarlabel='1 μm'
)

from skimage.draw import line as bresenham_line

r0, c0 = oversampling*4, oversampling*17   # row, col of p0                                                                                                                                       
r1, c1 = oversampling*12, oversampling*20  # row, col of p1
rows_1, cols_1 = bresenham_line(r0, c0, r1, c1)
values_1 = img1_subset[rows_1, cols_1]
# Coordinates of the line we'd like to sample along

ax.plot(cols_1, rows_1, color='white', linewidth=1.5)

r0, c0 = oversampling*(60-25), oversampling*(62+25)    # row, col of p0                                                                                                                                       
r1, c1 = oversampling*(64-25), oversampling*(72+25)  # row, col of p1
rows_2, cols_2 = bresenham_line(r0, c0, r1, c1)
values_2 = img1_subset[rows_2, cols_2]
# Coordinates of the line we'd like to sample along

ax.plot(cols_2, rows_2, color='white', linewidth=1.5, ls='--')


folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, 'Massive_Cells_Render_Zoomin.svg'), dpi=600, format='svg')
plt.show()

## Eps-sweep diagnostic (`Close_Red_Dyes.md` §14 §5, item 1)

The site-pooled unmixing above develops a real bimodal split, but the rendered result looks
*spatially segregated* into zones rather than interleaved structure — suspected artefact of
forcing one dye label per whole DBSCAN site, worse the more generously `epsilon_multiplier`
merges nearby (possibly different) structure into one site. Cheapest test before building a
bigger kinetic-first redesign: sweep `epsilon_multiplier` and render each independently — if
spatial segregation drops substantially at tighter eps, that's real evidence the effect is at
least partly a plain "clustering too coarse" problem, not necessarily requiring changepoint
segmentation + Bayesian per-site splitting to fix.

Each eps value gets its own independent figure below (not an overlay/animation) so they can be
compared side by side directly.

In [ ]:
EPS_MULTIPLIERS = [0.25, 0.5, 0.75, 1.0]

sweep_results = {}
for eps_mult in EPS_MULTIPLIERS:
    ch0, ch1, sw_sites, sw_diag = run_site_pooled_unmixing(
        linked_fremoved, epsilon_multiplier=eps_mult, verbose=True,
    )
    sweep_results[eps_mult] = {
        'channel_0': ch0, 'channel_1': ch1, 'sites': sw_sites, 'diagnostics': sw_diag,
    }


In [ ]:
plt.scatter(sweep_results[0.5]['channel_1']['xc'], sweep_results[0.5]['channel_1']['yc'])

In [ ]:
height

In [ ]:
# Render + plot each eps value as its own independent figure.
sweep_oversampling = 8
sweep_pixel_size = 72
sweep_min_size = 4 * (sweep_pixel_size / 1000)

sweep_info = [{
    "Width": width,
    "Height": height,
    "Frames": np.max(linked_fremoved['frame']),
    "Pixelsize": sweep_pixel_size,
}]

for eps_mult, res in sweep_results.items():
    _, sweep_img1 = render.render(
        res['channel_0'].to_records(index=False), sweep_info,
        oversampling=sweep_oversampling, blur_method='gaussian', min_blur_width=sweep_min_size,
    )
    _, sweep_img2 = render.render(
        res['channel_1'].to_records(index=False), sweep_info,
        oversampling=sweep_oversampling, blur_method='gaussian', min_blur_width=sweep_min_size,
    )

    fig, ax = plotter.one_column_plot(height=5, width=5)
    ax = plotter.multichannel_overlay_plot(
        ax, [sweep_img2, sweep_img1],
        cmaps=['hot', 'cyan'],
        pixelsize=(sweep_pixel_size / sweep_oversampling),
        scalebarsize=1000,
        vmins=[np.percentile(sweep_img2, 1), np.percentile(sweep_img1, 1)],
        vmaxs=[np.percentile(sweep_img2, 99.9), np.percentile(sweep_img1, 99.9)],
        brightness_boost=[1, 1],
        scalebarlabel='1 μm',
    )
    ax.set_title(
        f"epsilon_multiplier={eps_mult}  |  {res['diagnostics']['n_sites']:,} sites  |  "
        f"weights={np.round(res['diagnostics']['site_weights'], 2)}"
    )
    fig.savefig(f"eps_sweep_{eps_mult}.png", dpi=600)
    plt.show()


## HDBSCAN comparison (quick test, per user suggestion)

The eps sweep above didn't fix the spatial-segregation issue (cyan and orange never sit on
top of each other, when biologically they should) -- tightening `eps` just degrades
classification quality (0.25) without fixing segregation (0.75/1.0 look essentially the same).

DBSCAN uses one *global* radius everywhere. HDBSCAN instead builds a density hierarchy and
extracts the most stable clusters, adapting its effective merge radius to *local* density --
worth testing because this dataset has a real reason to expect locally-varying density (3x
imager-concentration difference between the two dyes, §7 item 4, means genuinely different
local revisit rates that no single global `eps` can be right for everywhere at once).

Same caveat as the eps sweep: this is still "cluster spatially → pool → one label per
cluster" -- if two real docking sites are close enough that even HDBSCAN's adaptive threshold
merges them, the majority-vote collapse still happens. Expect *somewhat* better, not a full
fix -- this is a cheap diagnostic, not a replacement for the kinetic-first redesign (§14).

In [ ]:
hdbscan_channel_0, hdbscan_channel_1, hdbscan_sites, hdbscan_diag = run_site_pooled_unmixing(
    linked_fremoved, min_cluster_size=5, clustering_method="hdbscan",
)


In [ ]:
# Render DBSCAN(eps=0.75, the best-looking eps-sweep point) and HDBSCAN side by
# side as independent figures, same convention as the eps sweep above.
compare_oversampling = 8
compare_pixel_size = 72
compare_min_size = 4 * (compare_pixel_size / 1000)
compare_info = [{
    "Width": width,
    "Height": height,
    "Frames": np.max(linked_fremoved['frame']),
    "Pixelsize": compare_pixel_size,
}]

compare_runs = {
    "dbscan (eps=0.75)": (sweep_results[0.75]['channel_0'], sweep_results[0.75]['channel_1'],
                           sweep_results[0.75]['diagnostics']),
    "hdbscan": (hdbscan_channel_0, hdbscan_channel_1, hdbscan_diag),
}

for label, (ch0, ch1, diag) in compare_runs.items():
    _, cmp_img1 = render.render(
        ch0.to_records(index=False), compare_info,
        oversampling=compare_oversampling, blur_method='gaussian', min_blur_width=compare_min_size,
    )
    _, cmp_img2 = render.render(
        ch1.to_records(index=False), compare_info,
        oversampling=compare_oversampling, blur_method='gaussian', min_blur_width=compare_min_size,
    )

    fig, ax = plotter.one_column_plot(height=5, width=5)
    ax = plotter.multichannel_overlay_plot(
        ax, [cmp_img2, cmp_img1],
        cmaps=['hot', 'cyan'],
        pixelsize=(compare_pixel_size / compare_oversampling),
        scalebarsize=10000,
        vmins=[np.percentile(cmp_img2, 1), np.percentile(cmp_img1, 1)],
        vmaxs=[np.percentile(cmp_img2, 99.9), np.percentile(cmp_img1, 99.9)],
        brightness_boost=[1, 1],
        scalebarlabel='10 μm',
    )
    ax.set_title(f"{label}  |  {diag['n_sites']:,} sites  |  weights={np.round(diag['site_weights'], 2)}")
    fig.savefig(f"compare_{label.split()[0]}.png", dpi=200)
    plt.show()


In [ ]:
def fwhm(sigma):
    return 2*np.sqrt(2*np.log(2))*sigma

In [ ]:
fig, axs = plotter.one_column_plot(npanels=2, ratios=[1,1], height=1.7, width=1)

dr, dc = np.diff(rows_1.astype(float)), np.diff(cols_1.astype(float))
distances_nm = np.concatenate([[0.], np.cumsum(np.sqrt(dr**2 + dc**2))]) * (69.0/oversampling)

bin_size_nm = 69.0 / 4  # e.g. 2 SR pixels per bar — adjust to taste                                                                                                     
                                                                                                                                                                          
bins = np.arange(distances_nm[0], distances_nm[-1] + bin_size_nm, bin_size_nm)
bin_counts, edges = np.histogram(distances_nm, bins=bins, weights=values_1)
bin_norm,   _     = np.histogram(distances_nm, bins=bins)            # pixels per bin
bin_means = np.where(bin_norm > 0, bin_counts / bin_norm, 0.0)       # mean intensity
bin_centres = (edges[:-1] + edges[1:]) / 2


axs[0].bar(bin_centres, bin_means, width=bin_size_nm * 0.75, color='#888', edgecolor='k', linewidth=0.4)
axs[0].set_xlim([0, 600])

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

def gaussian(x, A, mu, sigma):
  return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

def double_gaussian(x, A1, mu1, s1, A2, mu2, s2, offset):
  return gaussian(x, A1, mu1, s1) + gaussian(x, A2, mu2, s2) + offset

# Auto-guess peak positions from the binned data
peaks, props = find_peaks(bin_means, distance=3, prominence=bin_means.max()*0.2)

if len(peaks) >= 2:
  p1, p2 = peaks[np.argsort(props['prominences'])[-2:]]   # two tallest
  p0 = [
      bin_means[p1], bin_centres[p1], bin_size_nm * 2,
      bin_means[p2], bin_centres[p2], bin_size_nm * 2,
      0.0,
  ]
  try:
      popt, pcov = curve_fit(double_gaussian, bin_centres, bin_means, p0=p0,
                             bounds=(0, np.inf), maxfev=5000)
      perr = np.sqrt(np.diag(pcov))

      x_fit = np.linspace(bin_centres[0], bin_centres[-1], 500)
      fwhm_label = fwhm(popt[[2, 5]])
      separation_nm = abs(popt[4] - popt[1])
      axs[0].plot(0, 0, alpha=0, label=f'PS = {separation_nm:.1f} ± 'f'{np.sqrt(perr[1]**2 + perr[4]**2):.1f} nm')
      axs[0].plot(x_fit, gaussian(x_fit, *popt[[0,1,2]]) + popt[6], 'r--', linewidth=0.75, label='FWHM = '+str(int(fwhm_label[0]))+' nm')
      axs[0].plot(x_fit, gaussian(x_fit, *popt[[3,4,5]]) + popt[6], 'g--', linewidth=0.75, label='FWHM = '+str(int(fwhm_label[1]))+' nm')

  except RuntimeError:
      print("Fit did not converge — adjust p0 or bin_size_nm")

#axs[0].legend(loc='best', fontsize=6)
axs[0].set_xlabel('', fontsize=8)
axs[0].set_ylabel('', fontsize=8)
axs[0].set_yticklabels([])
axs[0].set_ylim([1, 30])
axs[0].grid(lw=0.5, alpha=0.5, ls='--', color='gray')

dr, dc = np.diff(rows_2.astype(float)), np.diff(cols_2.astype(float))
distances_nm = np.concatenate([[0.], np.cumsum(np.sqrt(dr**2 + dc**2))]) * (69.0/oversampling)


#########
bins = np.arange(distances_nm[0], distances_nm[-1] + bin_size_nm, bin_size_nm)
bin_counts, edges = np.histogram(distances_nm, bins=bins, weights=values_2)
bin_norm,   _     = np.histogram(distances_nm, bins=bins)            # pixels per bin
bin_means = np.where(bin_norm > 0, bin_counts / bin_norm, 0.0)       # mean intensity
bin_centres = (edges[:-1] + edges[1:]) / 2


axs[1].bar(bin_centres, bin_means, width=bin_size_nm * 0.75, color='#888', edgecolor='k', linewidth=0.4)
axs[1].set_xlim([0, 300])

# Auto-guess peak positions from the binned data
peaks, props = find_peaks(bin_means, distance=3, prominence=bin_means.max()*0.2)

if len(peaks) >= 2:
  p1, p2 = peaks[np.argsort(props['prominences'])[-2:]]   # two tallest
  p0 = [
      bin_means[p1], bin_centres[p1], bin_size_nm * 2,
      bin_means[p2], bin_centres[p2], bin_size_nm * 2,
      0.0,
  ]
  try:
      popt, pcov = curve_fit(double_gaussian, bin_centres, bin_means, p0=p0,
                             bounds=(0, np.inf), maxfev=5000)
      perr = np.sqrt(np.diag(pcov))

      x_fit = np.linspace(bin_centres[0], bin_centres[-1], 500)
      fwhm_label = fwhm(popt[[2, 5]])
      separation_nm = abs(popt[4] - popt[1])
      axs[1].plot(0, 0, alpha=0, label=f'PS = {separation_nm:.1f} ± 'f'{np.sqrt(perr[1]**2 + perr[4]**2):.1f} nm')
      axs[1].plot(x_fit, gaussian(x_fit, *popt[[0,1,2]]) + popt[6], 'r--', linewidth=0.75, label='FWHM = '+str(int(fwhm_label[0]))+' nm')
      axs[1].plot(x_fit, gaussian(x_fit, *popt[[3,4,5]]) + popt[6], 'g--', linewidth=0.75, label='FWHM = '+str(int(fwhm_label[1]))+' nm')

  except RuntimeError:
      print("Fit did not converge — adjust p0 or bin_size_nm")


#axs[1].legend(loc='best', fontsize=6)
axs[1].set_xlabel('distance / nm', fontsize=8)
axs[1].set_ylabel('', fontsize=8)
axs[1].set_yticklabels([])
axs[1].set_ylim([1, 100])
axs[1].grid(lw=0.5, alpha=0.5, ls='--', color='gray')

folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, 'Massive_Cells_Cutthroughs.svg'), dpi=600, format='svg')
plt.show()

In [ ]:
from pyS3M.FRCFunctions import fire


In [ ]:
# --- parameters ---
pixel_size_nm = 69      # nm per camera pixel
zoom          = 8      # SR pixels per camera pixel → 6.9 nm SR pixels

# linked_fremoved, width, height already defined in the notebook

# Shift localisations to start from (0,0) within their bounding box                                                                                                                                                                                            
red_xy = channel_0[['xc', 'yc']].to_numpy()                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                             
x_min, y_min = red_xy[:, 0].min(), red_xy[:, 1].min()     
red_xy_shifted = red_xy.copy()                                                                                                                                                                                                                                 
red_xy_shifted[:, 0] -= x_min                                                                                                                                                                                                                                  
red_xy_shifted[:, 1] -= y_min
                                                                                                                                                                                                                                                             
nx_eff_red = int(red_xy[:, 0].max() - x_min) + 1                                                                                                                                                                                                                   
ny_eff_red = int(red_xy[:, 1].max() - y_min) + 1                                                                                                                                                                                                                   
print(f"Effective field: {nx_eff_red} × {ny_eff_red} px  ({nx_eff_red*69:.0f} × {ny_eff_red*69:.0f} nm)")                                                                                                                                                                      
                                                                                                                                                                                                                                                             
resolution_nm_655, frc_mean_655, res_hi_nm_655, res_lo_nm_655 = fire(                                                                                                                                                                                          
  positions     = red_xy_shifted,                                                                                                                                                                                                                            
  nx            = nx_eff_red,                                                                                                                                                                                                                                    
  ny            = ny_eff_red,                                                                                                                                                                                                                                    
  zoom          = zoom,
  n_blocks      = 50,                                                                                                                                                                                                                                        
  pixel_size_nm = pixel_size_nm,                                                                                                                                                                                                                             
)                                                                                                                                                                                                                                                              
print(f"FIRE resolution: {resolution_nm_655:.1f} nm")    
print(f"FIRE bounds: {np.abs(res_hi_nm_655 - res_lo_nm_655):.1f} nm")                                                                                                                                                                                                          

In [ ]:
print(f"FIRE bounds: {np.abs(res_hi_nm_655 - res_lo_nm_655):.1f} nm")                                                                                                                                                                                                          

In [ ]:
# --- parameters ---
pixel_size_nm = 69      # nm per camera pixel
zoom          = 8      # SR pixels per camera pixel → 6.9 nm SR pixels

# linked_fremoved, width, height already defined in the notebook

green_xy = channel_1[['xc', 'yc']].to_numpy()

x_min, y_min = green_xy[:, 0].min(), green_xy[:, 1].min()     
green_xy_shifted = red_xy.copy()                                                                                                                                                                                                                                 
green_xy_shifted[:, 0] -= x_min                                                                                                                                                                                                                                  
green_xy_shifted[:, 1] -= y_min
                                                                                                                                                                                                                                                             
nx_eff_green = int(green_xy[:, 0].max() - x_min) + 1                                                                                                                                                                                                                   
ny_eff_green = int(green_xy[:, 1].max() - y_min) + 1                                                                                                                                                                                                                   
print(f"Effective field: {nx_eff_green} × {ny_eff_green} px  ({nx_eff_green*69:.0f} × {ny_eff_green*69:.0f} nm)")                                                                                                                                                                      
                                                                                                                                                                                                                                                             
resolution_nm_cy3B, frc_mean_cy3B, res_hi_nm_cy3B, res_lo_nm_cy3B = fire(                                                                                                                                                                                          
  positions     = green_xy_shifted,                                                                                                                                                                                                                            
  nx            = nx_eff_green,                                                                                                                                                                                                                                    
  ny            = ny_eff_green,                                                                                                                                                                                                                                    
  zoom          = zoom,
  n_blocks      = 50,                                                                                                                                                                                                                                        
  pixel_size_nm = pixel_size_nm,                                                                                                                                                                                                                             
)                                                                                                                                                                                                                                                              
print(f"FIRE resolution: {resolution_nm_cy3B:.1f} nm")   
print(f"FIRE bounds: {np.abs(res_hi_nm_cy3B - res_lo_nm_cy3B):.1f} nm")                                                                                                                                                                                                          

In [ ]:
print(f"FIRE bounds: {np.abs(res_hi_nm_cy3B - res_lo_nm_cy3B):.1f} nm")  

In [ ]:
fig, ax = plotter.one_column_plot(width=2.3, height=1.2)

sz_red       = max(nx_eff_red, ny_eff_red) * zoom
sz_green      = max(nx_eff_green, ny_eff_green) * zoom

sr_px_nm = pixel_size_nm / zoom        # nm per SR pixel

k_red   = np.arange(len(frc_mean_655))
q_red   = k_red / sz_red                           # cycles per SR pixel
q_nm_red = q_red / sr_px_nm                   # cycles per nm  (= 1 / d_nm)

k_green   = np.arange(len(frc_mean_cy3B))
q_green   = k_green / sz_green                           # cycles per SR pixel
q_nm_green = q_green / sr_px_nm                   # cycles per nm  (= 1 / d_nm)

# keep only below Nyquist
mask_red = (q_red > 0) & (q_red < 0.6)
mask_green = (q_green > 0) & (q_green < 0.6)

# 1/7 crossing in cycles/nm
q_cross_cy3B = 1.0 / resolution_nm_cy3B         # cycles per nm at the FIRE value
q_hi_cy3B    = 1.0 / res_hi_nm_cy3B
q_lo_cy3B    = 1.0 / res_lo_nm_cy3B


# 1/7 crossing in cycles/nm
q_cross_655 = 1.0 / resolution_nm_655         # cycles per nm at the FIRE value
q_hi_655    = 1.0 / res_hi_nm_655
q_lo_655    = 1.0 / res_lo_nm_655

ax = plotter.line_plot(ax, q_nm_red[mask_red], frc_mean_655[mask_red], label='FRC (mean), ATTO 655', color='darkred')
ax = plotter.line_plot(ax, q_nm_green[mask_green], frc_mean_cy3B[mask_green], label='FRC (mean), Cy3B', color='darkgreen')

ax.axhline(1/7, color='gray', ls='--', lw=1, label='1/7 threshold')
ax.axvline(q_cross_655, color='crimson', ls='-', lw=1,
           label=f'FIRE = {resolution_nm_655:.0f} nm')
ax.axvspan(q_lo_655, q_hi_655, alpha=0.15, color='crimson', label='±1σ')

ax.axvline(q_cross_cy3B, color='green', ls='-', lw=1,
           label=f'FIRE = {resolution_nm_cy3B:.0f} nm')
ax.axvspan(q_lo_cy3B, q_hi_cy3B, alpha=0.15, color='green', label='±1σ')

ax.set_xlabel(r'1 / spatial frequency (nm)', fontsize=8)
ax.set_ylabel('FRC', fontsize=8)
ax.set_ylim([0, 1.05])
ax.set_xscale('log')

tick_nm = [500, 100, 20]          # nm values to label
ax.set_xticks([1/d for d in tick_nm])

ax.set_xticklabels([f'{d} nm' for d in tick_nm])
ax.set_xlim(1/500, 1/15)
#ax.legend(fontsize=7)
plt.tight_layout()

folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, "FRC_MassiveCells.svg"), dpi=600, format="svg")


In [ ]:
# FRC in linear frequency space (cycles / nm)
fig, ax = plotter.one_column_plot(width=2.3, height=1.2)

ax = plotter.line_plot(ax, q_nm_red[mask_red],   frc_mean_655[mask_red],   label='FRC (mean), ATTO 655', color='darkred')
ax = plotter.line_plot(ax, q_nm_green[mask_green], frc_mean_cy3B[mask_green], label='FRC (mean), Cy3B',     color='darkgreen')

ax.axhline(1/7, color='gray', ls='--', lw=1, label='1/7 threshold')

ax.axvline(q_cross_655, color='crimson', ls='-', lw=1,
           label=f'FIRE = {resolution_nm_655:.0f} nm')
ax.axvspan(q_lo_655, q_hi_655, alpha=0.15, color='crimson', label='±1σ')

ax.axvline(q_cross_cy3B, color='green', ls='-', lw=1,
           label=f'FIRE = {resolution_nm_cy3B:.0f} nm')
ax.axvspan(q_lo_cy3B, q_hi_cy3B, alpha=0.15, color='green', label='±1σ')

ax.set_xlabel(r'spatial frequency / nm$^{-1}$', fontsize=8)
ax.set_ylabel('FRC', fontsize=8)
ax.set_ylim([0, 1.05])
ax.set_xlim(q_nm_red[mask_red][0], q_nm_red[mask_red][-1])

plt.tight_layout()

folder = '/scratch/sycamore-asap/2026_Multicolour_Paper/For_Talks'
plt.savefig(os.path.join(folder, "FRC_MassiveCells_LinearFreq.svg"), dpi=600, format="svg")
